In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import re
import spacy
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.layers import TextVectorization
from collections import Counter
import numpy as np

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

from tensorflow.keras.callbacks import TensorBoard

I0000 00:00:1776590839.005147   14591 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776590839.043498   14591 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776590840.230673   14591 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
df = pd.read_csv("../../asset/sentiment140_cleaned.csv")

In [3]:
df = df.dropna(subset=["text_nlp"]).reset_index(drop=True)

X = df["text_nlp"]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print(f"Train : {len(X_train):,} | Test : {len(X_test):,}")

Train : 1,270,277 | Test : 317,570


In [4]:
VOCAB_SIZE=15000
MAX_LEN=40
BATCH_SIZE=200_000

counter = Counter()
for i in range(0, len(X_train), BATCH_SIZE):
    batch = X_train[i : i + BATCH_SIZE]
    for text in batch:
        counter.update(text.split())
    print(f"Batch {i // BATCH_SIZE + 1} compté")

vocab = ['', '[UNK]'] + [word for word, _ in counter.most_common(VOCAB_SIZE - 2)]

vectorizer = TextVectorization(max_tokens=VOCAB_SIZE, output_sequence_length=MAX_LEN)
vectorizer.set_vocabulary(vocab)
print("Terminé !")

Batch 1 compté
Batch 2 compté
Batch 3 compté
Batch 4 compté
Batch 5 compté
Batch 6 compté
Batch 7 compté
Terminé !


E0000 00:00:1776590890.032305   14591 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [5]:
def vectorize_in_batches(data, vectorizer, batch_size):
    results = []
    total = len(data)
    
    for i in range(0, total, batch_size):
        batch = data[i : i + batch_size]
        vectorized = vectorizer(batch)
        results.append(vectorized.numpy())
        print(f"Batch {i // batch_size + 1}/{-(-total // batch_size)} traité — {min(i + batch_size, total):,} / {total:,}")
    
    return np.concatenate(results, axis=0)

X_train_vec = vectorize_in_batches(X_train, vectorizer, BATCH_SIZE)
X_test_vec  = vectorize_in_batches(X_test,  vectorizer, BATCH_SIZE)

print(f"X_train_vec shape : {X_train_vec.shape}")
print(f"X_test_vec shape  : {X_test_vec.shape}")

Batch 1/7 traité — 200,000 / 1,270,277
Batch 2/7 traité — 400,000 / 1,270,277
Batch 3/7 traité — 600,000 / 1,270,277
Batch 4/7 traité — 800,000 / 1,270,277
Batch 5/7 traité — 1,000,000 / 1,270,277
Batch 6/7 traité — 1,200,000 / 1,270,277
Batch 7/7 traité — 1,270,277 / 1,270,277
Batch 1/2 traité — 200,000 / 317,570
Batch 2/2 traité — 317,570 / 317,570
X_train_vec shape : (1270277, 40)
X_test_vec shape  : (317570, 40)


In [6]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

# ===== MODÈLE =====
model = Sequential([
    Embedding(VOCAB_SIZE, 128, mask_zero=True),
    Bidirectional(LSTM(128, return_sequences=True)),
    Bidirectional(LSTM(64)),
    Dropout(0.4),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# ===== CALLBACKS =====
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)

history = model.fit(
    X_train_vec, y_train,
    epochs=10,
    batch_size=512,
    validation_split=0.1,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/10


W0000 00:00:1776590946.510826   14591 cpu_allocator_impl.cc:82] Allocation of 365839680 exceeds 10% of free system memory.


2233/2233 ━━━━━━━━━━━━━━━━━━━━ 842s 375ms/step - accuracy: 0.7671 - loss: 0.4882 - val_accuracy: 0.7778 - val_loss: 0.4657 - learning_rate: 3.0000e-04
Epoch 2/10
2233/2233 ━━━━━━━━━━━━━━━━━━━━ 841s 376ms/step - accuracy: 0.7834 - loss: 0.4594 - val_accuracy: 0.7803 - val_loss: 0.4602 - learning_rate: 3.0000e-04
Epoch 3/10
2233/2233 ━━━━━━━━━━━━━━━━━━━━ 1230s 551ms/step - accuracy: 0.7904 - loss: 0.4464 - val_accuracy: 0.7829 - val_loss: 0.4579 - learning_rate: 3.0000e-04
Epoch 4/10
2233/2233 ━━━━━━━━━━━━━━━━━━━━ 969s 434ms/step - accuracy: 0.7963 - loss: 0.4364 - val_accuracy: 0.7840 - val_loss: 0.4584 - learning_rate: 3.0000e-04
Epoch 5/10
2233/2233 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8027 - loss: 0.4239
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0001500000071246177.
2233/2233 ━━━━━━━━━━━━━━━━━━━━ 4259s 2s/step - accuracy: 0.8012 - loss: 0.4270 - val_accuracy: 0.7824 - val_loss: 0.4634 - learning_rate: 3.0000e-04
Epoch 6/10
2233/2233 ━━━━━━━━━━━━━━━━━━━━ 851s 381

In [7]:
loss, accuracy = model.evaluate(X_test_vec, y_test, verbose=1)

9925/9925 ━━━━━━━━━━━━━━━━━━━━ 117s 12ms/step - accuracy: 0.7827 - loss: 0.4587


In [8]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = (model.predict(X_test_vec) > 0.5).astype(int)

print(classification_report(y_test, y_pred, target_names=['Négatif', 'Positif']))

9925/9925 ━━━━━━━━━━━━━━━━━━━━ 119s 12ms/step
              precision    recall  f1-score   support

     Négatif       0.79      0.77      0.78    158880
     Positif       0.77      0.80      0.79    158690

    accuracy                           0.78    317570
   macro avg       0.78      0.78      0.78    317570
weighted avg       0.78      0.78      0.78    317570

